# Supervised Classification with a Born-Machine MPS

We extend the MPS Born machine to **supervised classification** by appending
a dedicated label site to the path chain.

The joint MPS models `p(xi, y)` where `y ∈ {1, …, C}` is the class label.
Classification uses the **Born rule**:

$$p(y = c \mid \xi) \propto |\Psi(\xi_1, \ldots, \xi_M, c)|^2$$

The label site sits at the **end** of the chain (easiest integration); other
positions are explored in the notes.

### What we compare
- MPS Born classifier
- Logistic regression baseline

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.resolve()
Pkg.instantiate()

using MPSFast
using MPSFast.Encoders
using Random, LinearAlgebra, Statistics, Printf

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`
     Project No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Project.toml`
    Manifest No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Manifest.toml`
Precompiling packages...
   2168.6 ms  ✓ MPSFast
  1 dependency successfully precompiled in 3 seconds. 204 already precompiled.


## 1. Synthetic Dataset

Two classes: cumulative sums of ±1 random walks with different drift.
- Class 1: positive drift → path tends to rise.
- Class 2: negative drift → path tends to fall.

In [2]:
function synthetic_paths_labels(N, M; drift=0.15, σ=1.0, rng=Random.default_rng())
    paths  = Matrix{Float64}(undef, N, M)
    labels = Vector{Int}(undef, N)
    for i in 1:N
        y = rand(rng, 1:2)
        d = y == 1 ? drift : -drift
        s = cumsum(d .+ σ .* randn(rng, M))
        paths[i,  :] = s
        labels[i]    = y
    end
    return paths, labels
end

rng = MersenneTwister(42)
N_train, N_test, M = 2_000, 500, 15

train_paths, train_labels = synthetic_paths_labels(N_train, M; rng=rng)
test_paths,  test_labels  = synthetic_paths_labels(N_test,  M; rng=rng)

println("Train size: ", size(train_paths), "  class balance: ",
        count(==(1), train_labels), " / ", count(==(2), train_labels))
println("Test size : ", size(test_paths))

Train size: (2000, 15)  class balance: 972 / 1028
Test size : (500, 15)


## 2. Encode Labeled Paths

In [3]:
enc = BasisEncoder(3)   # d = 8 buckets
fit_grid!(enc, train_paths)

xi_train = encode_labeled_paths(enc, train_paths, train_labels; n_classes=2)
xi_test  = encode_labeled_paths(enc, test_paths,  test_labels;  n_classes=2)

Ml_class = classification_chain_length(enc, M)
d_class  = site_dim(enc)
n_classes = 2

println("Joint chain length  : ", Ml_class,  " (path sites + 1 label site)")
println("xi_train shape      : ", size(xi_train))
println("Label site values   : ", sort(unique(xi_train[:, end])))

Joint chain length  : 16 (path sites + 1 label site)
xi_train shape      : (2000, 16)
Label site values   : [1, 2]


## 3. Initialise Joint MPS

The label site has physical dimension `n_classes = 2`. We use a uniform
physical dimension `max(d_class, n_classes)` across all sites for simplicity
(the last site naturally uses only 1:n_classes).

In [4]:
D_max = 16

# Build MPS with label site of dim n_classes at the end
function init_mps_classification(Ml, d_path, n_classes, D_max; rng=Random.default_rng())
    T = Float32
    Ds = [1; [min(d_path^j, D_max) for j in 1:Ml-1]; 1]
    mps = [randn(rng, T, Ds[j], d_path, Ds[j+1]) for j in 1:Ml-1]
    # Last site: physical dim = n_classes
    push!(mps, randn(rng, T, Ds[Ml], n_classes, 1))
    left_canonicalize_mps!(mps)
    return mps
end

mps = init_mps_classification(Ml_class, d_class, n_classes, D_max; rng=MersenneTwister(1))
println("Joint MPS shapes: ", [size(mps[j]) for j in 1:Ml_class])

Joint MPS shapes: [(1, 8, 8), (8, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 8, 16), (16, 2, 1)]


## 4. Train Joint Born Machine

In [5]:
n_epochs = 20
η        = 5e-4
ε_cut    = 1e-5

nll_hist = train_mps!(
    mps, xi_train, n_epochs, η, D_max, ε_cut;
    verbose = true, nll_samples = 500,
)

train_mps!: Ml=16, Nd=2000, d=8, D_max=16, epochs=20, one-hot
— Epoch 1/20 —
  ↳ norm envs ready → forward sweep (15 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 1/20 | NLL ≈ 29.3282 | bonds=[8,16,16,16,16,16,16,16,16,16,16,16,16,16,2] | 0.168 s
— Epoch 2/20 —
  ↳ norm envs ready → forward sweep (15 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 2/20 | NLL ≈ 27.811 | bonds=[8,16,16,16,16,16,16,16,16,16,16,16,16,16,2] | 0.05 s
— Epoch 3/20 —
  ↳ norm envs ready → forward sweep (15 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 3/20 | NLL ≈ 26.7312 | bonds=[8,16,16,16,16,16,16,16,16,16,16,16,16,16,2] | 0.044 s
— Epoch 4/20 —
  ↳ norm envs ready → forward sweep (15 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …


20-element Vector{Float64}:
 29.328180825784845
 27.810977461794376
 26.73117816957629
 26.035988922190036
 25.39530838478852
 24.875034519557634
 24.27092177492376
 23.76341279107666
 23.27913075150932
 22.734174408511414
 22.50904008998983
 22.11536265659033
 21.604411684712986
 21.030852297381838
 20.870812434978728
 20.34094184256381
 20.295716674105424
 19.69482620367972
 19.44306571227661
 19.14790629932696

## 5. Evaluate Classification

In [6]:
# Training accuracy
acc_train = classification_accuracy(mps, xi_train, n_classes)
println("Train accuracy : ", round(100 * acc_train, digits=1), "%")

# Test accuracy
xi_test_path = encode_paths(enc, test_paths)  # path-only encoding for prediction
# Manually compute test accuracy using class_probabilities
correct_test = 0
for i in 1:N_test
    probs  = class_probabilities(mps, xi_test_path[i, :], n_classes)
    y_pred = argmax(probs)
    y_pred == test_labels[i] && (correct_test += 1)
end
acc_test = correct_test / N_test
println("Test  accuracy : ", round(100 * acc_test, digits=1), "%")

Train accuracy : 63.7%
Test  accuracy : 59.6%


In [7]:
# Show Born-rule probabilities on a few test paths
println("\nBorn probabilities on first 10 test paths:")
println("  i   True y   p(y=1)   p(y=2)   Pred")
for i in 1:10
    probs  = class_probabilities(mps, xi_test_path[i, :], n_classes)
    y_pred = argmax(probs)
    @printf "  %2d    %d      %.4f   %.4f    %d%s\n" i test_labels[i] probs[1] probs[2] y_pred (y_pred == test_labels[i] ? " ✓" : " ✗")
end


Born probabilities on first 10 test paths:
  i   True y   p(y=1)   p(y=2)   Pred
   1    1      0.9775   0.0225    1 ✓
   2    1      0.1244   0.8756    2 ✗
   3    1      0.4082   0.5918    2 ✗
   4    1      0.9479   0.0521    1 ✓
   5    2      0.6200   0.3800    1 ✗
   6    2      0.0304   0.9696    2 ✓
   7    2      0.1186   0.8814    2 ✓
   8    2      0.1115   0.8885    2 ✓
   9    1      0.7768   0.2232    1 ✓
  10    1      0.1675   0.8325    2 ✗


## 6. Logistic Regression Baseline

In [8]:
# Simple logistic regression on raw path features (last value + mean)
function logreg_features(paths)
    hcat(paths[:, end], mean(paths, dims=2)[:], std(paths, dims=2)[:])
end

function sigmoid(z); 1 ./ (1 .+ exp.(-z)); end

function logreg_train(X, y; lr=0.1, n_iter=500)
    n, p = size(X)
    w    = zeros(p); b = 0.0
    for _ in 1:n_iter
        logits = X * w .+ b
        phat   = sigmoid(logits)
        err    = phat .- (y .== 1)
        w     -= lr / n .* (X' * err)
        b     -= lr / n .* sum(err)
    end
    return w, b
end

X_train = logreg_features(train_paths)
X_test  = logreg_features(test_paths)

w, b    = logreg_train(X_train, train_labels)

function logreg_acc(X, y, w, b)
    preds = (sigmoid(X * w .+ b) .> 0.5) .+ 1
    mean(preds .== y)
end

lr_train = logreg_acc(X_train, train_labels, w, b)
lr_test  = logreg_acc(X_test,  test_labels,  w, b)

println("Logistic regression:")
println("  Train accuracy: ", round(100 * lr_train, digits=1), "%")
println("  Test  accuracy: ", round(100 * lr_test,  digits=1), "%")

println("\nMPS Born machine:")
println("  Train accuracy: ", round(100 * acc_train, digits=1), "%")
println("  Test  accuracy: ", round(100 * acc_test,  digits=1), "%")

Logistic regression:
  Train accuracy: 28.0%
  Test  accuracy: 25.2%

MPS Born machine:
  Train accuracy: 63.7%
  Test  accuracy: 59.6%


## 7. Discussion

The joint Born machine `p(xi, y)` is trained entirely via NLL on the
joint sequence `(xi_1, …, xi_M, y)` — no special classification objective.
Classification at inference time is a single forward pass computing
`|Ψ(xi, c)|²` for each `c`.

### Extensions to try
- Move the label site to position `M//2` or another location in the chain.
- Use `TrigEncoder` + Gram training for lower bond dimension.
- Multi-class: set `n_classes = 3` or more.
- Condition on a sub-sequence (partial observation) by marginalising over unknown sites.